# RQ2 dynamic-allocation pilot — seed 3, T4 x2

Runs exactly two 100-epoch methods concurrently: frozen `Geometry-Dynamic p=1` and `Resource-Dynamic matched-compute`. Both use the same model/data/loss/optimizer, four subnet forwards per batch, and validation-only evaluation. Attach the RQ2-v1 output, theory-allocation-probe output, and probabilistic-support-preview output.

In [ ]:
import os, subprocess, sys, time, json, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Missing Kaggle secret github_token'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN_RUNTIME':github_token})
try:
    command = ['git','-C',str(PROJECT_ROOT),'pull','--ff-only'] if (PROJECT_ROOT/'.git').is_dir() else ['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True); github_token = None
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'thop>=0.1.1'], check=True)
import torch
assert torch.cuda.device_count() == 2, f'Choose GPU T4 x2; detected {torch.cuda.device_count()}'
print('Commit:', subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip())
print([torch.cuda.get_device_name(i) for i in range(2)])

## Locate RQ2 development evidence and both frozen CPU-policy artifacts

In [ ]:
import importlib, rq2_anchor_placement, rq2_dynamic_seed3_pilot
rq2_anchor_placement = importlib.reload(rq2_anchor_placement)
rq2_dynamic_seed3_pilot = importlib.reload(rq2_dynamic_seed3_pilot)
RQ2_INPUT = Path('/kaggle/input/notebooks/dyhngg/test-rq2')
assert RQ2_INPUT.exists(), f'Attach RQ2-v1 output: {RQ2_INPUT}'
RQ2_ROOT = rq2_anchor_placement.find_rq2_development_root(RQ2_INPUT, '/kaggle/working/materialized-rq2-dynamic-pilot')
THEORY_ROOT, PREVIEW_ROOT = rq2_dynamic_seed3_pilot.find_frozen_policy_roots('/kaggle/input', '/kaggle/working/materialized-dynamic-policies')
print('RQ2 root:', RQ2_ROOT)
print('Frozen theory root:', THEORY_ROOT)
print('Frozen matched-resource preview root:', PREVIEW_ROOT)

## Run both seed-3 methods
Each method occupies one T4. The runner first performs 100k CPU pair draws, then trains 50+50 epochs with an optimizer/scheduler reset at epoch 51, saves epoch 50/100/latest checkpoints, and finally evaluates only the fixed validation split.

In [ ]:
import scripts.run_dynamic_seed3_pilot as dynamic_runner
dynamic_runner = importlib.reload(dynamic_runner)
RUN_DIR = Path('/kaggle/working/rq2-dynamic-seed3-pilot')
started = time.perf_counter()
result = dynamic_runner.run_seed3_pilot(RQ2_ROOT, THEORY_ROOT, PREVIEW_ROOT, RUN_DIR, gpu_ids=[0,1])
print(f'Pilot completed in {(time.perf_counter()-started)/3600:.2f} hours')
print(json.dumps(result['decision'], indent=2))

In [ ]:
import pandas as pd
from IPython.display import display
display(pd.read_csv(RUN_DIR/'pretraining_sampler_sanity.csv'))
display(pd.read_csv(RUN_DIR/'dynamic_pilot_method_summary.csv'))
display(pd.read_csv(RUN_DIR/'dynamic_pilot_width_comparison.csv'))
for method in ['geometry_dynamic','resource_dynamic']:
    history = pd.read_csv(RUN_DIR/method/'seed_3'/'training_epoch_metrics.csv')
    print(method, 'final cumulative realized/expected compute:', history.iloc[-1].cumulative_realized_total_flops_per_batch/history.iloc[-1].expected_total_flops_per_batch)

## Validate and export the resumable pilot bundle

In [ ]:
required = [RUN_DIR/'dynamic_pilot_decision.json', RUN_DIR/'dense_validation_accuracy.csv', RUN_DIR/'dynamic_pilot_method_summary.csv', RUN_DIR/'dynamic_pilot_width_comparison.csv']
for method in ['geometry_dynamic','resource_dynamic']:
    required += [RUN_DIR/method/'seed_3'/name for name in ['epoch_050.pt','epoch_100.pt','latest.pt','training_epoch_metrics.csv','width_inclusion_by_epoch.csv','pair_counts_by_epoch.csv','training_provenance.json']]
missing = [str(path) for path in required if not path.is_file() or path.stat().st_size == 0]
assert not missing, f'Missing pilot artifacts: {missing}'
bundle_path = Path('/kaggle/working/rq2-dynamic-seed3-pilot.zip')
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED, allowZip64=True) as bundle:
    for path in RUN_DIR.rglob('*'):
        if path.is_file() and path.name != 'checkpoint.pt':
            bundle.write(path, path.relative_to(RUN_DIR))
print('Download/persist:', bundle_path, f'{bundle_path.stat().st_size/2**30:.2f} GiB')
bundle_path